# Coreference Resolution: mention-ranking, transitive closure, and the metrics from scratch

Read *"John told his manager that he would finish the report by Friday. She thanked him."* and your brain effortlessly binds **his / he / him** to **John** and **She** to **the manager**. **Coreference resolution** is teaching a machine to do that: find every referring expression (mention) and **cluster** the ones that denote the same entity.

This notebook builds the core mechanics from scratch, one idea per cell, every cell **asserting its point before printing it**:

1. **mention-ranking** — score every candidate antecedent of a mention (including the dummy **ε = "no antecedent"**), softmax over them, pick the argmax, then form clusters by **transitive closure**;
2. the three coreference **metrics** — MUC (links), B³ (mentions), CEAF-φ4 (entity alignment) — and the **CoNLL F1** average, with B³ matching the by-hand derivation;
3. a **Winograd schema** — flip one adjective and the resolved antecedent flips, which world knowledge (not structure) decides;
4. real, reproducible **mention detection** with spaCy.

Everything is imported from the seeded source of truth `coreference.py`, so the notebook, the page, and the figures (`make_figures_14.py`) can never drift.

## Setup — import the seeded backend

The ranking math and the metrics are **pure-Python / numpy**, so the device is honestly CPU. The module seeds numpy once at import (deterministic runs) and uses a **stable md5 hash**, never Python's per-process-salted `hash()`. We print an honest runtime banner.

In [1]:
from coreference import runtime_banner, stable_hash

print(runtime_banner())
# stable across processes (md5), unlike Python's salted hash()
print(f"stable_hash('coreference') % 10**8 = {stable_hash('coreference') % 10**8:08d}")

device: cpu (pure-Python/numpy) | python 3.12.13 | numpy 2.4.6, torch 2.12.0 (not used here)
stable_hash('coreference') % 10**8 = 37349868


## The toy document and its mentions

Our running passage has **six** mentions across **two** entities: E1 = {John, his, he, him} (the person) and E2 = {his manager, She} (the manager). Each mention carries the cues a linear scorer needs — gender, number, whether it is a discourse subject, and whether it is a pronoun (pronouns are anaphoric; full NPs usually *introduce* an entity).

In [2]:
from coreference import MENTIONS, gold_clusters

assert len(MENTIONS) == 6
assert [m['text'] for m in MENTIONS] == ['John', 'his', 'his manager', 'he', 'She', 'him']
# the gold clustering has exactly two entities
gold = gold_clusters()
assert len(gold) == 2 and sorted(map(len, gold)) == [2, 4]
print('mentions:', [m['text'] for m in MENTIONS])
print('gold clusters (by index):', gold)
print('  E1 =', [MENTIONS[i]['text'] for i in gold[0]])
print('  E2 =', [MENTIONS[i]['text'] for i in gold[1]])

mentions: ['John', 'his', 'his manager', 'he', 'She', 'him']
gold clusters (by index): [[0, 1, 3, 5], [2, 4]]
  E1 = ['John', 'his', 'he', 'him']
  E2 = ['his manager', 'She']


## Mention-ranking 1 — score the candidate antecedents of one mention

For mention $i$, the candidate set is $\mathcal{Y}(i) = \{\epsilon, 1, \ldots, i-1\}$ — the dummy **ε** ("no antecedent / new entity") plus every earlier mention. A real-valued score $s(i,j)$ ranks them. ε is pinned at **0**, so a real candidate only wins if it scores *above* 0 — an elegant built-in new-entity threshold.

Resolve **him** (the last mention): gender agreement should reward the masculine candidates (John, his, he) and crush the feminine *She*.

In [3]:
from coreference import EPSILON, candidate_scores

him_index = 5  # 'him'
cands = candidate_scores(MENTIONS, him_index)
scored = [(EPSILON if k == EPSILON else MENTIONS[k]['text'], round(s, 2)) for k, s in cands]
# the masculine, recent, subject 'he' must outscore every other candidate, incl. epsilon (0.0)
best_key = max(cands, key=lambda kv: kv[1])[0]
assert best_key != EPSILON and MENTIONS[best_key]['text'] == 'he'
# the feminine 'She' must score below epsilon (gender clash)
she_score = next(s for k, s in cands if k != EPSILON and MENTIONS[k]['text'] == 'She')
assert she_score < 0.0
print('scores for antecedents of "him":')
for name, s in scored:
    print(f'  {name:<12} s(i,j) = {s:+.2f}')

scores for antecedents of "him":
  ε            s(i,j) = +0.00
  John         s(i,j) = +2.70
  his          s(i,j) = +2.25
  his manager  s(i,j) = +0.33
  he           s(i,j) = +3.00
  She          s(i,j) = -1.50


## Mention-ranking 2 — softmax over the candidates

Turn the scores into a distribution: $P(y_i = j) = \dfrac{e^{s(i,j)}}{\sum_{j'} e^{s(i,j')}}$ over the candidate antecedents including ε. The argmax is the predicted antecedent. These are the *exact* numbers the page's Worked Example 2 walks through by hand (he 0.43, John 0.32, his 0.20, his manager 0.03, ε 0.02, She 0.005) — table, figure, and notebook are one computation.

In [4]:
import numpy as np
from coreference import antecedent_distribution

keys, probs = antecedent_distribution(MENTIONS, him_index)
labels = [EPSILON if k == EPSILON else MENTIONS[k]['text'] for k in keys]
# it is a valid probability distribution, and 'he' carries the most mass
assert abs(probs.sum() - 1.0) < 1e-9
assert labels[int(probs.argmax())] == 'he'
# lock the EXACT probabilities the page's Worked Example 2 table claims (page == figure == notebook)
prob_by_label = {lab: round(float(p), 2) for lab, p in zip(labels, probs)}
assert prob_by_label['he'] == 0.43
assert prob_by_label['John'] == 0.32
assert prob_by_label['his'] == 0.20
assert prob_by_label['his manager'] == 0.03
assert prob_by_label[EPSILON] == 0.02
assert round(prob_by_label['She'], 3) == 0.0  # 0.005 -> rounds to 0.00 at 2 dp; ~0.005 at 3 dp
for lab, p in sorted(zip(labels, probs), key=lambda lp: -lp[1]):
    print(f'  P(antecedent = {lab:<12}) = {p:.3f}')

  P(antecedent = he          ) = 0.427
  P(antecedent = John        ) = 0.316
  P(antecedent = his         ) = 0.202
  P(antecedent = his manager ) = 0.030
  P(antecedent = ε           ) = 0.021
  P(antecedent = She         ) = 0.005


## Mention-ranking 3 — predict antecedents for every mention

Run the same softmax-argmax for **each** mention. A pronoun should point back to its referent; a new name or full NP should pick **ε** (start a new entity).

In [5]:
from coreference import predict_antecedents

preds = predict_antecedents(MENTIONS)
# the two entity-starting mentions (John, his manager) must choose epsilon
assert preds[0] == EPSILON  # John starts E1
assert preds[2] == EPSILON  # 'his manager' starts E2
for i, a in enumerate(preds):
    ant = EPSILON if a == EPSILON else MENTIONS[a]['text']
    print(f'  {MENTIONS[i]["text"]:<12} -> antecedent: {ant}')

  John         -> antecedent: ε
  his          -> antecedent: John
  his manager  -> antecedent: ε
  he           -> antecedent: John
  She          -> antecedent: his manager
  him          -> antecedent: he


## Mention-ranking 4 — transitive closure into clusters

Antecedent links form a forest; the **transitive closure** (union-find) recovers the clusters: if $a \to b$ and $b \to c$, then $\{a,b,c\}$ is one entity. *Link locally, cluster globally.*

In [6]:
from coreference import resolve

result = resolve()
# the toy model recovers the two gold entities exactly -> perfect CoNLL F1 on this passage
assert result['clusters_text'] == [['John', 'his', 'he', 'him'], ['his manager', 'She']]
assert abs(result['conll_f1'] - 1.0) < 1e-9
print('predicted clusters:')
for c in result['clusters_text']:
    print('  ', c)
print(f'CoNLL F1 vs gold on the toy passage: {result["conll_f1"]:.3f}')

predicted clusters:
   ['John', 'his', 'he', 'him']
   ['his manager', 'She']
CoNLL F1 vs gold on the toy passage: 1.000


## Evaluation 1 — B³ from scratch, matching the page by hand

Coreference output is a **clustering**, so there is no single metric — the field averages three. **B³** (Bagga & Baldwin 1998) scores **per mention**: for mention $m$ with gold cluster $G_m$ and predicted cluster $P_m$,
$$\text{precision}(m) = \frac{|G_m \cap P_m|}{|P_m|}, \qquad \text{recall}(m) = \frac{|G_m \cap P_m|}{|G_m|},$$
then average. The page's example splits mention `c` out of the {a,b,c} entity; the by-hand answer is P=1.0, R=0.733, F1=0.846.

In [7]:
from coreference import METRIC_GOLD, METRIC_PRED, bcubed

bP, bR, bF = bcubed(METRIC_GOLD, METRIC_PRED)
# matches the by-hand table in the page exactly
assert (round(bP, 3), round(bR, 3), round(bF, 3)) == (1.0, 0.733, 0.846)
print('gold:', METRIC_GOLD)
print('pred:', METRIC_PRED, '  (mention c wrongly split off)')
print(f'B-cubed  P={bP:.3f}  R={bR:.3f}  F1={bF:.3f}')

gold: [['a', 'b', 'c'], ['d', 'e']]
pred: [['a', 'b'], ['c'], ['d', 'e']]   (mention c wrongly split off)
B-cubed  P=1.000  R=0.733  F1=0.846


## Evaluation 2 — MUC, CEAF-φ4, and the CoNLL average

The **same error** lands on a **different number** under each metric — because each is sensitive to a different failure mode (links vs mentions vs entity alignment). The CoNLL-2012 score is their unweighted mean.

In [8]:
from coreference import muc, ceaf_phi4, conll_f1

mP, mR, mF = muc(METRIC_GOLD, METRIC_PRED)
cP, cR, cF = ceaf_phi4(METRIC_GOLD, METRIC_PRED)
conll = conll_f1(METRIC_GOLD, METRIC_PRED)
# the three F1s genuinely disagree on the same error, and CoNLL is their mean
assert round(mF, 2) == 0.80 and round(cF, 2) == 0.72
assert abs(conll - (mF + bF + cF) / 3) < 1e-9
print(f'MUC       P={mP:.3f} R={mR:.3f} F1={mF:.3f}')
print(f'B-cubed   P={bP:.3f} R={bR:.3f} F1={bF:.3f}')
print(f'CEAF-phi4 P={cP:.3f} R={cR:.3f} F1={cF:.3f}')
print(f'CoNLL avg F1 = {conll:.3f}')

MUC       P=1.000 R=0.667 F1=0.800
B-cubed   P=1.000 R=0.733 F1=0.846
CEAF-phi4 P=0.600 R=0.900 F1=0.720
CoNLL avg F1 = 0.789


## Evaluation 3 — B³ precision punishes over-merging; recall punishes over-splitting

The two ways any clustering can be wrong. Gold = one 4-mention entity. **Over-split** it into two halves → recall drops to 0.5. **Over-merge** two gold entities into one → precision drops to 0.5. Same F1 (0.667), opposite diagnosis — which is why you report P and R, not just F1.

In [9]:
split_P, split_R, split_F = bcubed([['w', 'x', 'y', 'z']], [['w', 'x'], ['y', 'z']])
merge_P, merge_R, merge_F = bcubed([['w', 'x'], ['y', 'z']], [['w', 'x', 'y', 'z']])
# over-split: perfect precision, halved recall; over-merge is the mirror image
assert (round(split_P, 3), round(split_R, 3)) == (1.0, 0.5)
assert (round(merge_P, 3), round(merge_R, 3)) == (0.5, 1.0)
assert abs(split_F - merge_F) < 1e-9  # same F1, opposite failure
print(f'over-SPLIT : P={split_P:.2f} R={split_R:.2f} F1={split_F:.3f}  (recall punished)')
print(f'over-MERGE : P={merge_P:.2f} R={merge_R:.2f} F1={merge_F:.3f}  (precision punished)')

over-SPLIT : P=1.00 R=0.50 F1=0.667  (recall punished)
over-MERGE : P=0.50 R=1.00 F1=0.667  (precision punished)


## Winograd schema — why world knowledge is needed

*The trophy doesn't fit in the suitcase because **it** is too **big**.* → *it* = the **trophy**.
Flip one word: *...too **small**.* → *it* = the **suitcase**. Same syntax, same agreement (both singular/neuter). Only **commonsense** about containment decides — flip the adjective, flip the antecedent.

In [10]:
from coreference import winograd_resolve

big_winner, big_scores = winograd_resolve('big')
small_winner, small_scores = winograd_resolve('small')
# the resolved antecedent FLIPS when only the adjective changes
assert big_winner == 'trophy' and small_winner == 'suitcase'
# without world knowledge, structure/agreement give both candidates an equal score (a tie)
_, tie_scores = winograd_resolve('big', use_world_knowledge=False)
assert tie_scores['trophy'] == tie_scores['suitcase']
print(f"'...because it is too big.'    -> it = {big_winner}   {big_scores}")
print(f"'...because it is too small.'  -> it = {small_winner} {small_scores}")
print(f"without world knowledge        -> tie {tie_scores} (structure alone cannot decide)")

'...because it is too big.'    -> it = trophy   {'trophy': 1.0, 'suitcase': -1.0}
'...because it is too small.'  -> it = suitcase {'trophy': -1.0, 'suitcase': 1.0}
without world knowledge        -> tie {'trophy': 0.0, 'suitcase': 0.0} (structure alone cannot decide)


## Real mention detection — spaCy on the running passage

The ranking demo used hand-specified mentions for clarity; here is **real, reproducible** mention detection. spaCy gives named entities + pronouns + noun chunks — the candidate spans a linker would then resolve. The neural coref *models* (`fastcoref`, `maverick-coref`) are pinned to older transformer versions and break on current installs, so production paths today are a pinned env or an LLM prompt — but the **mention detection** and **metrics** are exactly what we compute here.

In [11]:
from coreference import spacy_mentions

det = spacy_mentions()
# the noun-chunk + pronoun list matches the by-hand worked example on the page
assert ('John', 'PERSON') in det['ner']
assert det['pronouns'] == ['his', 'he', 'She', 'him']
assert 'his manager' in det['noun_chunks'] and 'the update' in det['noun_chunks']
print('NER         :', det['ner'])
print('pronouns    :', det['pronouns'])
print('noun chunks :', det['noun_chunks'])

NER         : [('John', 'PERSON'), ('Friday', 'DATE')]
pronouns    : ['his', 'he', 'She', 'him']
noun chunks : ['John', 'his manager', 'he', 'the report', 'Friday', 'She', 'him', 'the update']


## Recap

- **Mention-ranking** runs **one softmax per mention** over its candidate antecedents (including **ε**), picks the argmax, and **transitive closure** chains the links into clusters. ε pinned at 0 is the built-in "start a new entity" threshold.
- Coreference needs **three metrics** — MUC counts links (blind to singletons), B³ counts mentions (singleton-safe), CEAF aligns entities (punishes the wrong cluster *count*) — averaged into **CoNLL F1**. B³ **precision punishes over-merging**, **recall punishes over-splitting**.
- **Winograd schemas** can't be cracked by syntax/agreement — only **world knowledge** — which is exactly where LLMs leapt ahead.

Every number here is computed by `coreference.py`; the chapter figures are regenerated from the same functions by `make_figures_14.py`.